### Feature engineering

In [10]:
import torch 

# --- Configuration & Split Windows ---
TRAIN_YM = [202502, 202503, 202504, 202505, 202506]  # Feb - Jun
TEST_YM  = [202507, 202508]                          # Jul - Aug
MIN_QDATE = "2025-02-01"                             # Drop pre-window stragglers

_GPU = torch.cuda.is_available()
DEVICE = torch.device("cuda" if _GPU else "cpu")
XGB_DEV = "cuda" if _GPU else "cpu"
CB_TASK = "GPU" if _GPU else "CPU"
LGBM_DEV = "cpu"

# Feature Schema
SUB_CAT = ["Group", "Owner", "PomsLauncher", "CampaignName", "CampaignStageName", 
           "CampaignId", "CampaignStageId", "CampaignType", "TestLaunch", "JobsubGroup", 
           "Image", "BlacklistSites"]
SUB_NUM = ["RequestCpus", "RequestDisk", "RequestMemory", "RequestSlots", 
           "ExecutableSize", "TransferInputMB", "ExpectedLifetime", "TotalSubmitProcs"]
MATCH_CAT = ["MatchSite", "MatchEntry", "MatchQueue", "MatchResource", "MatchCpus"]
MATCH_NUM = ["CpusProvisioned", "DiskProvisioned", "MemoryProvisioned"]
CAT_ALL = SUB_CAT + MATCH_CAT
NUM_ALL = SUB_NUM + MATCH_NUM

# Standard column mapping
CAND = {
    "Group": ["Group", "AccountingGroup"], "Owner": ["Owner"],
    "PomsLauncher": ["POMS_LAUNCHER", "POMS4_LAUNCHER"], "CampaignName": ["POMS4_CAMPAIGN_NAME"],
    "CampaignStageName": ["POMS4_CAMPAIGN_STAGE_NAME"], "CampaignId": ["POMS4_CAMPAIGN_ID"],
    "CampaignStageId": ["POMS4_CAMPAIGN_STAGE_ID"],
    "CampaignType": ["DerivedCampaignType"],
    "TestLaunch": ["POMS4_TEST_LAUNCH"], "JobsubGroup": ["Jobsub_Group"],
    "Image": ["SingularityImage"], "BlacklistSites": ["Blacklist_Sites"],
    "MatchSite": ["MATCH_EXP_JOB_GLIDEIN_Site", "MATCH_GLIDEIN_Site"],
    "MatchEntry": ["MATCH_GLIDEIN_Entry_Name"], "MatchQueue": ["MATCH_GLIDEIN_SiteWMS_Queue"],
    "MatchResource": ["MachineAttrGLIDEIN_ResourceName0"], "MatchCpus": ["MachineAttrCpus0"],
    "Node": ["LastRemoteHost", "RemoteHost"],
    "RequestCpus": ["RequestCpus"], "RequestDisk": ["RequestDisk"], "RequestMemory": ["RequestMemory"],
    "RequestSlots": ["RequestSlots"], "ExecutableSize": ["ExecutableSize"],
    "TransferInputMB": ["TransferInputSizeMB"], "ExpectedLifetime": ["JOB_EXPECTED_MAX_LIFETIME"],
    "TotalSubmitProcs": ["TotalSubmitProcs"], "CpusProvisioned": ["CpusProvisioned"],
    "DiskProvisioned": ["DiskProvisioned"], "MemoryProvisioned": ["MemoryProvisioned"],
    "QDate": ["QDate_ms", "QDate"],
    "JobStart": ["JobStartDate_ms", "JobStartDate", "JobCurrentStartDate_ms", "JobCurrentStartDate"],
    "CompletionDate": ["CompletionDate_ms", "CompletionDate"],
}

### Deriving CampaignType from stage name

`POMS4_CAMPAIGN_TYPE` is null for every row in the raw data (100% -- verified
against all 32 partition files), including the ~25% of jobs that have real
`POMS4_CAMPAIGN_ID`/`NAME`/`STAGE_NAME` values. It was never populated
upstream; this isn't something earlier cells in this notebook broke.

Per the "Campaign Type" table in `FIFE-Docs.md`, the type can be recovered
from keywords in `POMS4_CAMPAIGN_STAGE_NAME` instead. Jobs with no POMS4_*
fields at all aren't part of any campaign -- these get `"User"`.

In [11]:
# Campaign-type keyword table, mirroring the "Campaign Type" table in
# FIFE-Docs.md. Edit here (and keep the doc in sync) to reclassify a
# stage-name substring.
CAMPAIGN_TYPE_KEYWORDS = {
    "Generation": ["gen", "dio", "endpoint", "corsika", "sim", "wiremod", "ly", "offset",
                   "spill", "g4", "beamgun", "decay", "surface", "cryo"],
    "Reconstruction": ["stage0", "stage1", "reco", "reco1", "reco2", "digi", "track", "decode",
                       "recluster", "fullproduction", "ndlar", "compress", "convert", "fmatch"],
    "Merging": ["merge", "skim", "hadd", "filter", "scrub", "watchdog", "sleep", "test",
                "fclless", "concat", "mix"],
    "Analysis": ["ana", "caf", "ntuple", "larcv"],
}


def _classify_campaign_type(stage_name):
    """Composite stage names (e.g. 'gen_g4_detsim_reco1_reco2_caf') match
    keywords from several categories at once. We take the LAST (rightmost)
    match, on the assumption that the terminal step is what the stage
    actually delivers -- that example ends in 'caf' -> Analysis, even though
    it runs gen/reco steps first. This tie-break decides ~2.98M jobs across
    45 composite stage names, so it's worth a sanity check against how
    these have been labeled by hand in the past.
    """
    low = stage_name.lower()
    best_cat, best_pos = None, -1
    for cat, kws in CAMPAIGN_TYPE_KEYWORDS.items():
        for kw in kws:
            pos = low.find(kw)
            if pos > best_pos:
                best_pos, best_cat = pos, cat
    return best_cat or "Unmapped"


# Build the lookup once over the distinct stage names (~164), then map the
# whole column against it -- far cheaper than a per-row Python call.
_distinct_stages = (
    lf.select(pl.col("POMS4_CAMPAIGN_STAGE_NAME").unique())
    .collect()["POMS4_CAMPAIGN_STAGE_NAME"].drop_nulls().to_list()
)
_stage_to_type = {s: _classify_campaign_type(s) for s in _distinct_stages}

# No POMS4 fields at all -> not part of a campaign -> a user job.
lf = lf.with_columns(
    pl.when(pl.col("POMS4_CAMPAIGN_STAGE_NAME").is_null())
      .then(pl.lit("User"))
      .otherwise(
          pl.col("POMS4_CAMPAIGN_STAGE_NAME").replace_strict(_stage_to_type, default="Unmapped")
      )
      .alias("DerivedCampaignType")
)

# Register the derived column in `have`, the same way cell 2 does for
# "Group". `have` is what CAND/cat_expr() gate on -- a column present in
# `lf` but missing from `have` is silently replaced by a constant 0, which
# is exactly the all-null CampaignType failure this cell exists to fix.
if "DerivedCampaignType" not in have:
    have.append("DerivedCampaignType")

_diag = lf.group_by("DerivedCampaignType").len().sort("len", descending=True).collect()
_total = _diag["len"].sum()
print("CampaignType distribution:")
for row in _diag.iter_rows(named=True):
    print(f"  {row['DerivedCampaignType']:15s} {row['len']:>12,}  ({100 * row['len'] / _total:5.2f}%)")

# Recover the integer code each label ends up with. cat_expr() hashes the
# string, then cell 17 dense-ranks via np.unique -- which sorts -- so a
# label's final code is the rank of its hash among the hashes of the labels
# actually present. Built from _diag (observed values) rather than a fixed
# list, so it stays correct if a filter ever drops a type entirely.
_present = _diag["DerivedCampaignType"].to_list()
_hashes = (
    pl.DataFrame({"v": _present})
    .select(
        (pl.col("v").cast(pl.Utf8).fill_null("__NA__").hash(seed=0) % 2147483629)
        .cast(pl.Int32).alias("h")
    )["h"].to_list()
)
CAMPAIGN_TYPE_CODES = {
    rank: lab
    for rank, (lab, _) in enumerate(sorted(zip(_present, _hashes), key=lambda kv: kv[1]))
}
print("\nCampaignType code -> label (saved to schema_meta.json):")
for _code, _lab in CAMPAIGN_TYPE_CODES.items():
    print(f"  {_code} -> {_lab}")

_unmapped = sorted(s for s, t in _stage_to_type.items() if t == "Unmapped")
assert not _unmapped, (
    f"{len(_unmapped)} stage name(s) matched no keyword: {_unmapped}. "
    f"Extend CAMPAIGN_TYPE_KEYWORDS above (and FIFE-Docs.md) to cover them."
)
print(f"\nAll {len(_distinct_stages)} distinct stage names mapped; "
      f"{_diag.height} unique CampaignType values.")


CampaignType distribution:
  User              48,042,514  (75.04%)
  Reconstruction     9,225,360  (14.41%)
  Analysis           5,223,471  ( 8.16%)
  Generation         1,462,717  ( 2.28%)
  Merging               71,013  ( 0.11%)

CampaignType code -> label (saved to schema_meta.json):
  0 -> Reconstruction
  1 -> Analysis
  2 -> User
  3 -> Merging
  4 -> Generation

All 164 distinct stage names mapped; 5 unique CampaignType values.


In [12]:
import os
import gc
import sys
import time
import datetime as _dt
import numpy as np
import polars as pl
import torch

def cat_expr(std):
    raw = R.get(std)
    if raw is None or raw not in have:
        return pl.lit(0).cast(pl.Int32).alias("c_" + std)
    return ((pl.col(raw).cast(pl.Utf8).fill_null("__NA__").hash(seed=0) % 2147483629)
            .cast(pl.Int32).alias("c_" + std))

def num_expr(std):
    raw = R.get(std)
    if raw is None or raw not in have:
        return pl.lit(0.0).cast(pl.Float32).alias("n_" + std)
    return pl.col(raw).cast(pl.Float32, strict=False).fill_null(0.0).alias("n_" + std)

R = {std: next((c for c in cands if c in have), None) for std, cands in CAND.items()}

sel_exprs = (
    [cat_expr(c) for c in CAT_ALL] + [cat_expr("Node")]
    + [num_expr(c) for c in NUM_ALL]
    + [to_sec(R["QDate"]).alias("t_q") if R["QDate"] else pl.lit(None).alias("t_q"),
       to_sec(R["JobStart"]).alias("t_s") if R["JobStart"] else pl.lit(None).alias("t_s"),
       to_sec(R["CompletionDate"]).alias("t_c") if R["CompletionDate"] else pl.lit(None).alias("t_c")]
    + [pl.col("Failed").cast(pl.Int8), pl.col("fault_type").cast(pl.Int8), pl.col("wait_s").cast(pl.Float64)]
)

t0 = time.time()
QMIN_S = _dt.datetime.fromisoformat(MIN_QDATE).replace(tzinfo=_dt.timezone.utc).timestamp()
df = lf.filter(to_sec(R["QDate"]) >= QMIN_S).select(sel_exprs).collect()
ntot = df.height
print(f"Loaded {ntot:,} rows into RAM in {time.time() - t0:.1f}s")

# Extract categorical codes
codes = np.column_stack([df["c_" + c].to_numpy() for c in CAT_ALL])
node_k = df["c_Node"].to_numpy()
cards = []
for j in range(codes.shape[1]):
    u, inv = np.unique(codes[:, j], return_inverse=True)
    codes[:, j] = inv.astype(np.int32)
    cards.append(len(u))

# Extract numerical features
X_num = np.column_stack([df["n_" + c].to_numpy() for c in NUM_ALL]).astype(np.float32)
for j in range(X_num.shape[1]):
    s = float(X_num[:, j].std()) or 1.0
    X_num[:, j] = (X_num[:, j] - X_num[:, j].mean()) / s

# Timestamps & Labels
qs = df["t_q"].to_numpy()
jst = df["t_s"].to_numpy()
js = jst  # Alias for start time
comp = df["t_c"].to_numpy()
wait_sv = df["wait_s"].to_numpy()

#failed = df["Failed"].to_numpy()
ftype = df["fault_type"].to_numpy()
failed = (ftype >= 0).astype(np.int8)
hw = (ftype == 1).astype(np.int8)

# Month split masking
dtq = pl.from_epoch(pl.Series(np.nan_to_num(qs).astype(np.int64)), time_unit="s")
ym = (dtq.dt.year().cast(pl.Int32) * 100 + dtq.dt.month().cast(pl.Int32)).to_numpy()
tr_mask = np.isin(ym, TRAIN_YM)
te_mask = np.isin(ym, TEST_YM)

print(f"{ntot:,} total rows | Train ({TRAIN_YM}): {tr_mask.sum():,} | Test ({TEST_YM}): {te_mask.sum():,}")

Loaded 64,025,075 rows into RAM in 28.1s
64,025,075 total rows | Train ([202502, 202503, 202504, 202505, 202506]): 48,546,633 | Test ([202507, 202508]): 15,478,442


In [52]:
TRAIL_WINDOWS = [900, 3600]          # 15 min & 1 hour trailing windows
TRAIL_NAMES = ["site_fail", "entry_fail", "node_fail", "site_hw", "node_hw", "camp_fail"]

n_cat, n_num = len(CAT_ALL), len(NUM_ALL)
n_base = n_cat + n_num
n_sc, n_sn = len(SUB_CAT), len(SUB_NUM)

XMATCH_COLS = (CAT_ALL + [c + " (std)" for c in NUM_ALL]
               + ["sin_hour@match", "cos_hour@match", "sin_wday@match", "cos_wday@match"]
               + [f"trail{w // 60}m_{nm}" for w in TRAIL_WINDOWS for nm in TRAIL_NAMES]
               + ["log_site_running@match", "log_total_running@match"])

XSUB_COLS = (SUB_CAT + [c + " (std)" for c in SUB_NUM]
             + ["sin_hour@submit", "cos_hour@submit", "sin_wday@submit", "cos_wday@submit"]
             + ["log_idle_queue_depth", "log_camp_trail_wait", "log_total_running@submit"])

NXM, NXS = len(XMATCH_COLS), len(XSUB_COLS)

# Vectorized helper for trailing failure/hardware rates
def trailing_rate(key, tref, tcomp, outcome, is_comp, window_s):
    n = len(key); w = int(window_s)
    ci = np.where(is_comp)[0]
    ck = np.asarray(key)[ci].astype(np.int64)
    ct = np.rint(np.asarray(tcomp)[ci]).astype(np.int64)
    co = np.asarray(outcome)[ci].astype(np.float64)
    if len(ct) == 0:
        return np.full(n, np.nan, np.float32), np.zeros(n, np.float32)
    o = np.lexsort((ct, ck)); ck, ct, co = ck[o], ct[o], co[o]
    csum = np.concatenate([[0.0], np.cumsum(co)])
    t0 = int(ct.min()); span = int(ct.max()) - t0 + w + 3
    comp_key = ck * span + (ct - t0)
    tref = np.asarray(tref); tcomp = np.asarray(tcomp); outcome = np.asarray(outcome)
    tq = np.where(np.isnan(tref), t0 - w - 10, np.rint(tref)).astype(np.int64)
    kq = np.asarray(key).astype(np.int64)
    hi = np.searchsorted(comp_key, kq * span + np.clip(tq - t0, -1, span - 2), "right")
    lo = np.searchsorted(comp_key, kq * span + np.clip(tq - w - t0, -1, span - 2), "right")
    cnt = (hi - lo).astype(np.float64); ssum = csum[hi] - csum[lo]
    self_in = np.asarray(is_comp) & ~np.isnan(tref) & (tcomp <= tref) & (tcomp > tref - window_s)
    ssum[self_in] -= outcome[self_in]; cnt[self_in] -= 1
    with np.errstate(invalid="ignore", divide="ignore"):
        rate = np.where(cnt > 0, ssum / np.maximum(cnt, 1), np.nan)
    return rate.astype(np.float32), cnt.astype(np.float32)

# Cyclical clock features helper
def cyc_of(t):
    d = pl.from_epoch(pl.Series(np.asarray(t)).cast(pl.Int64, strict=False), time_unit="s")
    h = d.dt.hour().to_numpy().astype(np.float64)
    dw = d.dt.weekday().to_numpy().astype(np.float64)
    return np.nan_to_num(np.column_stack([np.sin(2 * np.pi * h / 24), np.cos(2 * np.pi * h / 24),
                                          np.sin(2 * np.pi * dw / 7), np.cos(2 * np.pi * dw / 7)])).astype(np.float32)

print("Calculating trailing state features...")
comp_ok = ~np.isnan(np.asarray(comp)); H1 = 3600.0
fl64, hw64 = np.asarray(failed).astype(np.float64), hw.astype(np.float64)
jsf = np.where(np.isnan(np.asarray(js)), -1e18, np.asarray(js))
tref_m = np.where(np.isnan(np.asarray(js)), np.asarray(qs), np.asarray(js))

site_k = codes[:, CAT_ALL.index("MatchSite")]
entry_k = codes[:, CAT_ALL.index("MatchEntry")]
camp_k = codes[:, CAT_ALL.index("CampaignId")]

trail_specs = [(site_k, jsf, fl64), (entry_k, jsf, fl64), (node_k, jsf, fl64),
               (site_k, jsf, hw64), (node_k, jsf, hw64), (camp_k, qs, fl64)]

tr_all = []
for w in TRAIL_WINDOWS:
    for kv, tv, ov in trail_specs:
        r, _ = trailing_rate(kv, tv, comp, ov, comp_ok, float(w))
        tr_all.append(np.nan_to_num(r))

print("Calculating concurrency features...")
fin2 = ~np.isnan(np.asarray(jst)) & ~np.isnan(np.asarray(comp))
s2 = np.sort(np.asarray(jst)[fin2]); c2 = np.sort(np.asarray(comp)[fin2])

def running_total(t):
    r = (np.searchsorted(s2, t, "right") - np.searchsorted(c2, t, "right")).astype(np.float64)
    self_run = fin2 & (np.asarray(jst) <= t) & (np.asarray(comp) > t)
    r[self_run] -= 1
    return np.clip(r, 0, None)

def running_by_key(key, t):
    ki = np.asarray(key).astype(np.int64)
    ks = ki[fin2]
    st = np.rint(np.asarray(jst)[fin2]).astype(np.int64)
    ct = np.rint(np.asarray(comp)[fin2]).astype(np.int64)
    t0 = int(min(st.min(), ct.min())); span = int(max(st.max(), ct.max())) - t0 + 3
    a_s = np.sort(ks * span + (st - t0)); a_c = np.sort(ks * span + (ct - t0))
    q = ki * span + np.clip(np.rint(t).astype(np.int64) - t0, -1, span - 2)
    r = (np.searchsorted(a_s, q, "right") - np.searchsorted(a_c, q, "right")).astype(np.float64)
    self_run = fin2 & (np.asarray(jst) <= t) & (np.asarray(comp) > t)
    r[self_run] -= 1
    return np.clip(r, 0, None)

run_site = running_by_key(site_k, tref_m)
run_tot_m = running_total(tref_m)
cycS = cyc_of(tref_m)

# Assemble Xmatch in RAM
print("Assembling Xmatch matrix in RAM...")
Xmatch = np.empty((int(ntot), NXM), dtype=np.float32)
Xmatch[:, :n_cat] = codes
Xmatch[:, n_cat:n_base] = X_num
Xmatch[:, n_base:n_base + 4] = cycS
for j in range(len(tr_all)):
    Xmatch[:, n_base + 4 + j] = tr_all[j]
Xmatch[:, NXM - 2] = np.log1p(run_site)
Xmatch[:, NXM - 1] = np.log1p(run_tot_m)
del cycS, tr_all, run_site, run_tot_m

# Queue depth & submission features
print("Calculating submission & queue features...")
fin = ~np.isnan(np.asarray(jst))
q_fin = np.sort(np.asarray(qs)[fin]); s_fin = np.sort(np.asarray(jst)[fin])
idle = np.clip(np.searchsorted(q_fin, np.asarray(qs), "right")
               - np.searchsorted(s_fin, np.asarray(qs), "right"), 0, None).astype(np.float64)
wok = comp_ok & ~np.isnan(np.asarray(wait_sv))
campw, _ = trailing_rate(camp_k, qs, comp, np.nan_to_num(np.asarray(wait_sv)), wok, H1)
run_tot_q = running_total(np.asarray(qs))
cycQ = cyc_of(qs)

# Assemble Xsub in RAM
print("Assembling Xsub matrix in RAM...")
Xsub = np.empty((int(ntot), NXS), dtype=np.float32)
Xsub[:, :n_sc] = codes[:, :n_sc]
Xsub[:, n_sc:n_sc + n_sn] = X_num[:, :n_sn]
Xsub[:, n_sc + n_sn:n_sc + n_sn + 4] = cycQ
Xsub[:, -3] = np.log1p(idle)
Xsub[:, -2] = np.log1p(np.nan_to_num(campw))
Xsub[:, -1] = np.log1p(run_tot_q)
del cycQ, idle, campw, run_tot_q, q_fin, s_fin, jsf, fl64, hw64
gc.collect()

# Zero-copy views
X = Xmatch[:, :n_base]
trail = Xmatch[:, n_base + 4:NXM - 2]
cyc_q = Xsub[:, n_sc + n_sn:n_sc + n_sn + 4]

print(f"Xmatch shape: {Xmatch.shape}  (base {n_base} + cyc 4 + trailing {6 * len(TRAIL_WINDOWS)} + concurrency 2)")
print(f"Xsub shape:   {Xsub.shape}  (sub cat {n_sc} + sub num {n_sn} + cyc 4 + queue state 3)")

Calculating trailing state features...
Calculating concurrency features...
Assembling Xmatch matrix in RAM...
Calculating submission & queue features...
Assembling Xsub matrix in RAM...
Xmatch shape: (64025075, 46)  (base 28 + cyc 4 + trailing 12 + concurrency 2)
Xsub shape:   (64025075, 27)  (sub cat 12 + sub num 8 + cyc 4 + queue state 3)


### Saving full data

In [33]:
output_path = "/mnt/scratch/fast0/amaustin/datasets/fife_batch_queues_anon_02-09_2025.parquet"

lf.sink_parquet(
    output_path,
    compression="zstd",
    compression_level=10, 
)
print(f"Dataset successfully saved to {output_path}")

Dataset successfully saved to /mnt/scratch/fast0/amaustin/datasets/fife_batch_queues_anon_02-09_2025.parquet


### Saving training data for later runs (skip)

In [14]:
import os 
import json
import numpy as np

SAVE_DIR = "/mnt/scratch/fast0/amaustin/datasets/fife/"
os.makedirs(SAVE_DIR, exist_ok=True)

# 1. Save target arrays individually for fast loading
np.save(os.path.join(SAVE_DIR, "failed.npy"), failed)
np.save(os.path.join(SAVE_DIR, "fault_type.npy"), ftype)
np.save(os.path.join(SAVE_DIR, "hw.npy"), hw)

# 2. Update the targets in the .npz file (without saving Xmatch/Xsub)
np.savez_compressed(
    os.path.join(SAVE_DIR, "targets_and_masks.npz"),
    failed=failed,
    hw=hw,
    wait_sv=wait_sv,
    tr_mask=tr_mask,
    te_mask=te_mask,
    qs=qs,
    jst=jst,
    comp=comp,
    fault_type=ftype
)

# 3. Sanity check directly from disk
saved_failed = np.load(os.path.join(SAVE_DIR, "failed.npy"))
print(f"Saved failed.npy successfully.")
print(f"Total rows       : {len(saved_failed):,}")
print(f"Genuine Failures : {saved_failed.sum():,} ({saved_failed.mean()*100:.2f}%)")

Saved failed.npy successfully.
Total rows       : 64,025,075
Genuine Failures : 8,989,483 (14.04%)
Saving training data to /mnt/scratch/fast0/amaustin/datasets/fife/


Saving metadata

In [ ]:
metadata = {
    "XMATCH_COLS": XMATCH_COLS,
    "XSUB_COLS": XSUB_COLS,
    "cards": [int(c) for c in cards],
    "NCAT_MATCH": len(CAT_ALL),
    "NCAT_SUB": len(SUB_CAT),
    "n_base": n_base,
}

# Code -> label for CampaignType (JSON object keys must be strings). The
# codes come from a hash + dense-rank, so they carry no inherent meaning --
# without this map the column is unreadable downstream.
if "CAMPAIGN_TYPE_CODES" in dir():
    metadata["CAMPAIGN_TYPE_CODES"] = {str(k): v for k, v in CAMPAIGN_TYPE_CODES.items()}
else:
    print("[!] CAMPAIGN_TYPE_CODES not defined -- run the CampaignType derivation cell.")

with open(os.path.join(SAVE_DIR, "schema_meta.json"), "w") as f:
    json.dump(metadata, f, indent=2)